# Team 04 Tool Dev Notebook

This notebook is the deterministic development harness for the active Team 04 Python tool surface.

Use it when changing geometry helpers, tool schemas, placement heuristics, or graph-backed shape serialization before involving the live LLM path.

## Why this notebook exists

The current runtime already follows the supervision pattern well: planner -> central_reason -> tool spokes.

The practical improvement here is workflow separation:
1. deterministic tool-development in this notebook
2. live-LLM end-to-end validation in the companion notebook

The next code-level improvement after this split is a wing-targeted edit tool layer keyed by the stable graph indices.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

workspace_root = Path.cwd().resolve()
candidate_roots = (
    workspace_root,
    workspace_root.parent,
    workspace_root / "team_04",
    workspace_root.parent / "team_04",
)
TEAM_ROOT = next((path for path in candidate_roots if (path / "agent").exists()), None)
if TEAM_ROOT is None:
    raise FileNotFoundError(
        "Run this notebook from the workspace root, the team_04 folder, or the team_04/test_notebooks folder."
    )

team_root_str = str(TEAM_ROOT)
if team_root_str not in sys.path:
    sys.path.insert(0, team_root_str)

DEV_PAYLOAD_PATH = TEAM_ROOT / "test_notebooks" / "tool_dev_mode_payload.json"
DEV_PAYLOAD_PATH

WindowsPath('C:/Users/baoqt/OneDrive/Documents/GitHub/AIA26_Studio/team_04/test_notebooks/tool_dev_mode_payload.json')

In [ ]:
from agent.mcp_client import build_default_local_tool_client
from agent.tool_catalog import ToolCatalog
from agent.tools.generate_building_boundary import generate_building_boundary
from agent.tools.modify_building_boundary import modify_building_boundary

from topologicpy.Edge import Edge
from topologicpy.Face import Face
from topologicpy.Graph import Graph
from topologicpy.Plotly import Plotly
from topologicpy.Vertex import Vertex

tool_client = build_default_local_tool_client()
catalog = ToolCatalog.from_discovered_tools(tool_client.list_tools())

action_names = (
    "read_site",
    "generate_shape",
    "check_requested_position",
    "check_constraints",
    "optimize",
    "evaluate",
    "place_building",
    "analyze_remaining_positions",
)

{action: list(catalog.names_for_action(action)) for action in action_names}

{'read_site': [],
 'generate_shape': ['generate_building_boundary'],
 'check_requested_position': ['remaining_buildable_positions',
  'requested_position_checker'],
 'check_constraints': [],
 'optimize': ['modify_building_boundary'],
 'evaluate': [],
 'place_building': ['import_building_boundary'],
 'analyze_remaining_positions': ['remaining_buildable_positions',
  'requested_position_checker']}

In [ ]:
def _to_xyz(point):
    if len(point) >= 3:
        return (float(point[0]), float(point[1]), float(point[2]))
    return (float(point[0]), float(point[1]), 0.0)


def _face_from_boundary(boundary):
    vertices = [Vertex.ByCoordinates(*_to_xyz(point)) for point in boundary[:-1]]
    return Face.ByVertices(vertices)


def _graph_from_wings(wings, edges):
    vertices = [
        Vertex.ByCoordinates(*_to_xyz(wing["centroid"]))
        for wing in wings
    ]
    graph_edges = [
        Edge.ByVertices(vertices[item["from_wing_index"]], vertices[item["to_wing_index"]])
        for item in edges
    ]
    return Graph.ByVerticesEdges(vertices, graph_edges)


def make_plan_figure(site_boundary, boundary, wings, *, title):
    site_face = _face_from_boundary(site_boundary)
    building_face = _face_from_boundary(boundary)
    wing_faces = [_face_from_boundary(wing["boundary"]) for wing in wings]

    plot_data = []
    plot_data += Plotly.DataByTopology(
        site_face,
        showVertices=False,
        showFaces=False,
        edgeColor="#2563eb",
        edgeWidth=5,
    )
    plot_data += Plotly.DataByTopology(
        building_face,
        showVertices=False,
        faceColor="#f97316",
        faceOpacity=0.2,
        edgeColor="#ea580c",
        edgeWidth=4,
    )

    wing_palette = ["#ef4444", "#22c55e", "#8b5cf6", "#f59e0b", "#06b6d4"]
    for index, wing_face in enumerate(wing_faces):
        plot_data += Plotly.DataByTopology(
            wing_face,
            showVertices=False,
            faceColor=wing_palette[index % len(wing_palette)],
            faceOpacity=0.35,
            edgeColor=wing_palette[index % len(wing_palette)],
            edgeWidth=2,
        )

    figure = Plotly.FigureByData(plot_data, width=950, height=650)
    figure.update_layout(title=title)
    return figure


def make_graph_figure(wings, building_graph, *, title):
    graph = _graph_from_wings(wings, building_graph["edges"])
    plot_data = Plotly.DataByGraph(
        graph,
        vertexColor="#111827",
        vertexSize=16,
        edgeColor="#0f766e",
        edgeWidth=4,
        showVertexLegend=False,
        showEdgeLegend=False,
    )
    figure = Plotly.FigureByData(plot_data, width=700, height=500)
    figure.update_layout(title=title)
    return figure

In [3]:
SITE_BOUNDARY = [
    [0.0, 0.0, 0.0],
    [72.0, 0.0, 0.0],
    [72.0, 48.0, 0.0],
    [0.0, 48.0, 0.0],
    [0.0, 0.0, 0.0],
]

generation_result = generate_building_boundary(
    area=900.0,
    building_type="U",
    building_depth=18.0,
    shape_ratio=0.62,
    site_boundary=SITE_BOUNDARY,
    optimize_placement=True,
    placement_clearance=2.0,
    population_size=40,
    generation_count=40,
    random_seed=11,
)

DEV_PAYLOAD_PATH.write_text(json.dumps(generation_result, indent=2), encoding="utf-8")
data = generation_result["data"]

{
    "saved_to": str(DEV_PAYLOAD_PATH),
    "shape_type": data["shape_type"],
    "wing_count": len(data["wings"]),
    "adjacency_list": data["building_graph"]["adjacency_list"],
    "placement_optimization": data["placement_optimization"],
    "site_fit_summary": data["site_fit_summary"],
}

{'saved_to': 'C:\\Users\\baoqt\\OneDrive\\Documents\\GitHub\\AIA26_Studio\\team_04\\test_notebooks\\tool_dev_mode_payload.json',
 'shape_type': 'U',
 'wing_count': 3,
 'adjacency_list': [[1, 2], [0], [0]],
 'placement_optimization': {'optimized': True,
  'centroid_xy': [35.327894, 24.232804],
  'rotation_degrees': 179.999972,
  'objective': -12.828074,
  'outside_area_sqm': 0.0,
  'clearance_m': 12.828074,
  'fits_within_site_boundary': True,
  'population_size': 40,
  'generation_count': 40,
  'random_seed': 11,
  'target_location_xy': []},
 'site_fit_summary': {'outside_area_sqm': 0.0,
  'clearance_m': 12.828074,
  'fits_within_site_boundary': True,
  'clearance_target_m': 2.0}}

In [ ]:
make_graph_figure(
    data["wings"],
    data["building_graph"],
    title="Wing graph and adjacency for the generated building",
)

In [ ]:
make_plan_figure(
    SITE_BOUNDARY,
    data["boundary"],
    data["wings"],
    title="Generated footprint inside the site boundary",
)

In [4]:
modified_result = modify_building_boundary(
    geometry_id=data["geometry_id"],
    boundary=data["boundary"],
    translate_by_xy=(4.0, -3.0),
    rotation_degrees=15.0,
    site_boundary=SITE_BOUNDARY,
    clearance=1.5,
)

{
    "geometry_id": modified_result["data"]["geometry_id"],
    "transformed_centroid": modified_result["data"]["centroid"],
    "fits_within_site_boundary": modified_result["data"]["fits_within_site_boundary"],
    "violations": modified_result["data"]["violations"],
    "transform_parameters": modified_result["data"]["transform_parameters"],
}

{'geometry_id': 'generate_building_boundary_08c4bf024608',
 'transformed_centroid': [39.327894, 21.232804, 0.0],
 'fits_within_site_boundary': True,
 'violations': [],
 'transform_parameters': {'target_centroid_xy': None,
  'translate_by_xy': [4.0, -3.0],
  'rotation_degrees': 15.0,
  'orientation_degrees': 0.0,
  'applied_rotation_degrees': 15.0,
  'rotation_origin_xy': [35.327894, 24.232804],
  'apply_mirror': False,
  'mirror_axis': 'y',
  'clearance': 1.5}}

In [ ]:
make_plan_figure(
    SITE_BOUNDARY,
    modified_result["data"]["transformed_boundary"],
    data["wings"],
    title="Transformed footprint inside the site boundary",
)

## Next use

Use this notebook when you are iterating on tool contracts, geometry math, graph payloads, or placement heuristics.

Use the companion end-to-end notebook when you want the real planner plus supervisor plus LLM path.